# Check: projected Zeeman matrices $M_\alpha \propto \sigma_\alpha$ and signs of $g_{ii}$

Top valence Kramers pair of the 2H-Ge nanowire (wire axis $x$, cross-section $y$-$z$), at **one electric-field strength, $|\mathcal{E}| = 6$ V/µm**, for several field directions in the $y$-$z$ plane.

**Method**

1. Build $H_0$ (B = 0) with kwant, add the electrostatic potential $V = -\mathcal{E}\cdot(\mathbf r - L/2)$, and get the Kramers pair $P = (|1\rangle, |2\rangle)$ by shift-invert `eigsh`.
2. Fix **one** pseudospin basis for all three field directions: `psi_new_basis` ($S_z$ diagonal in the pair, $\langle 1|S_x|2\rangle$ real and positive), then, with `SPIN_UP_FIRST = True`, column 0 = the state with **larger** $\langle S_z\rangle$ (pseudospin $\Uparrow$ ≈ spin up). `psi_new_basis` alone (as in `EDSR/M_matrix/export_M*.py`) puts the lower $\langle S_z\rangle$ first. Swapping the two columns ($P\to P\sigma_x$) flips $g_{\alpha y}$ and $g_{\alpha z}$ and leaves the physics (and $\det G$) unchanged.
3. For $\alpha = x, y, z$:
   $$M_\alpha = P^\dagger \frac{\partial H}{\partial B_\alpha} P = \frac{\mu_B}{2}\sum_\beta g_{\alpha\beta}\,\sigma_\beta, \qquad g_{\alpha\beta} = \frac{\mathrm{Tr}(M_\alpha \sigma_\beta)}{\mu_B},$$
   where $\partial H/\partial B_\alpha$ = orbital part (central difference, Peierls / $k_x$ shift) + spin part $g_s\mu_B S_\alpha$.
   - $B_x$: $\mathbf A = (0,0,B y)$, Peierls phases on the $y$-$z$ hoppings
   - $B_y$: $\mathbf A = (B z,0,0)$, $k_x \to k_x + eBz/\hbar$ at the site / bond midpoint
   - $B_z$: $\mathbf A = (-B y,0,0)$, $k_x \to k_x - eBy/\hbar$ at the site / bond midpoint
4. Checks:
   - **$M_\alpha \propto \sigma_\alpha$ in this basis**: the off-diagonal elements $g_{\alpha\beta}$ ($\beta\neq\alpha$) should be $\ll |g_{\alpha\alpha}|$, and $\mathrm{Tr}\,M_\alpha = 0$.
   - **Basis-independent version**: the $g$-vectors $\mathbf g_\alpha = (g_{\alpha x}, g_{\alpha y}, g_{\alpha z})$ are mutually orthogonal, i.e. $G G^T$ is diagonal, so $x, y, z$ are principal axes and *some* pseudospin basis makes every $M_\alpha \propto \sigma_\alpha$.
   - **Signs**: individual signs of $g_{ii}$ depend on the pseudospin convention (a $\pi$ rotation of the pseudospin flips two of them). The invariant is $\operatorname{sign}\det G$ (= $\operatorname{sign}(g_{xx}g_{yy}g_{zz})$ when $G$ is diagonal). Both are recorded.


In [1]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "4")

import time
import numpy as np
import kwant
from scipy.sparse.linalg import eigsh
from Hamiltonian_mathematica_v2 import gso, spin_matrix_small, apply_spin_operator, psi_new_basis

np.set_printoptions(precision=5, suppress=True, linewidth=120)

In [2]:
# ---------------- parameters ----------------
N = int(os.environ.get("GF_N", 100))   # sites per direction (N = 100 matches EDSR/import_data)
Ny = Nz = N
L = 300.0             # Angstrom
Nband = 10
kx = 0.0
Bf = 0.01             # T, finite-difference step (orbital term)
muB = 5.788e-5        # eV/T
gspin = 2.0
hbar = 1.05457e-34
h = 6.626e-34
q = 1.602e-19
sigma_val = 0.12      # eV, shift-invert target
N_EIG = 4             # Kramers pair + next pair (gap check)

E_V_um = 6.0          # |E| (V/um)
# direction: theta (deg) measured from +z toward +y  (0 -> E_z, 90 -> E_y)
THETAS_DEG = [90.0, 0.0, 83.2]
# cases (label, |E| in V/um, theta in deg); E = 0 is the reference
CASES = [("E = 0", 0.0, 0.0)] + [({90.0: "E || y", 0.0: "E || z"}.get(th, f"theta={th}"), E_V_um, th)
                                  for th in THETAS_DEG]
SPIN_UP_FIRST = True   # column 0 = larger <S_z>  (False -> raw psi_new_basis, = EDSR export convention)

a = L / (Ny + 1)
SX, SY, SZ = spin_matrix_small()
S_SPIN = {"x": SX, "y": SY, "z": SZ}
PAULI = {"x": np.array([[0, 1], [1, 0]], complex),
         "y": np.array([[0, -1j], [1j, 0]], complex),
         "z": np.array([[1, 0], [0, -1]], complex)}
AX = ["x", "y", "z"]
print(f"N = {N}, a = {a:.3f} A, Hilbert-space dim = {N*N*Nband}")

N = 100, a = 2.970 A, Hilbert-space dim = 100000


## Orbital Hamiltonians for $\pm B_\alpha$

$k_x$ shift for $B_y$, $B_z$ evaluated at the site (onsite) or the bond midpoint (hopping), uniformly for every block. This is the corrected convention of `EDSR/M_matrix/Hamiltonian_mathematica_v2.py`. It is equivalent to `gs_shifted_by/bz` in this folder called with the tail-site coordinate.

In [3]:
def kx_shift(B, n):
    # e B (n a) / hbar  in 1/Angstrom  (n in lattice units)
    return 2 * np.pi / a * n * B * (a * 1e-10) ** 2 / (h / q)


def make_system(field, B):
    yc = (Ny - 1) / 2.0
    zc = (Nz - 1) / 2.0
    blocks0 = {(dy, dz): np.asarray(gso(a, kx, dy, dz), complex)
               for dy in (-1, 0, 1) for dz in (-1, 0, 1)}
    pref = q * B * (a * 1e-10) ** 2 / hbar

    def onsite(site):
        y, z = site.tag
        if field == "y" and B != 0:
            return np.asarray(gso(a, kx + kx_shift(B, z - zc), 0, 0), complex)
        if field == "z" and B != 0:
            return np.asarray(gso(a, kx - kx_shift(B, y - yc), 0, 0), complex)
        return blocks0[(0, 0)]

    def hopping(s1, s2):
        y1, z1 = s1.tag
        y2, z2 = s2.tag
        dy, dz = int(y1 - y2), int(z1 - z2)
        if field == "x" and B != 0:          # A = (0, 0, B y): Peierls phase
            ymid = 0.5 * (y1 + y2) - yc
            return blocks0[(dy, dz)] * np.exp(-1j * pref * ymid * dz)
        if field == "y" and B != 0:          # A = (B z, 0, 0)
            zmid = 0.5 * (z1 + z2) - zc
            return np.asarray(gso(a, kx + kx_shift(B, zmid), dy, dz), complex)
        if field == "z" and B != 0:          # A = (-B y, 0, 0)
            ymid = 0.5 * (y1 + y2) - yc
            return np.asarray(gso(a, kx - kx_shift(B, ymid), dy, dz), complex)
        return blocks0[(dy, dz)]

    syst = kwant.Builder()
    lat = kwant.lattice.square(norbs=Nband)
    syst[(lat(y, z) for y in range(Ny) for z in range(Nz))] = onsite
    for d in [(1, 0), (0, 1), (1, 1), (1, -1)]:
        syst[kwant.builder.HoppingKind(d, lat, lat)] = hopping
    return syst.finalized()


def tags_of(fs):
    return np.array([s.tag for s in fs.sites], float)


t0 = time.time()
f0 = make_system(None, 0.0)
TAGS = tags_of(f0)
H0 = f0.hamiltonian_submatrix(sparse=True).tocsr()
dH_orb = {}
for ax in AX:
    fp, fm = make_system(ax, +Bf), make_system(ax, -Bf)
    assert np.array_equal(tags_of(fp), TAGS) and np.array_equal(tags_of(fm), TAGS)
    dH_orb[ax] = ((fp.hamiltonian_submatrix(sparse=True)
                   - fm.hamiltonian_submatrix(sparse=True)) / (2 * Bf)).tocsr()
    print(f"dH/dB_{ax} built,  ||dH - dH^dag|| = {abs(dH_orb[ax] - dH_orb[ax].getH()).max():.1e}")
print(f"systems built in {time.time() - t0:.0f} s")

dH/dB_x built,  ||dH - dH^dag|| = 0.0e+00


dH/dB_y built,  ||dH - dH^dag|| = 0.0e+00


dH/dB_z built,  ||dH - dH^dag|| = 0.0e+00
systems built in 14 s


## Kramers pair, common pseudospin basis, $M_x, M_y, M_z$

In [4]:
def potential_diagonal(Ef, theta):
    ypos = (TAGS[:, 0] + 1) * a - L / 2
    zpos = (TAGS[:, 1] + 1) * a - L / 2
    return np.repeat(-Ef * (np.sin(theta) * ypos + np.cos(theta) * zpos), Nband)


def zeeman_matrices(E_Vum, theta_deg):
    Ef = E_Vum * 1e-4                                    # V/Angstrom
    H = H0.copy()
    H.setdiag(H.diagonal() + potential_diagonal(Ef, np.deg2rad(theta_deg)))
    w, v = eigsh(H, k=N_EIG, sigma=sigma_val, which="LM")
    order = np.argsort(np.abs(w - sigma_val))
    pair, rest = order[:2], order[2:]
    info = dict(E_pair=w[pair].mean(), split0=abs(w[pair[0]] - w[pair[1]]),
                gap_next=np.min(np.abs(w[rest][:, None] - w[pair][None, :])))
    P, _ = np.linalg.qr(v[:, pair])                      # orthonormalize inside the pair
    p1, p2 = psi_new_basis(P[:, 0], P[:, 1])       # ONE basis for x, y, z
    if SPIN_UP_FIRST:                                    # P -> P sigma_x
        p1, p2 = p2, p1
    P = np.column_stack((p1, p2))
    info["P"] = P                                       # kept for the Rashba check
    info["Sz_pair"] = [np.vdot(p, apply_spin_operator(p, SZ)).real for p in (p1, p2)]

    M = {}
    for ax in AX:
        DP = np.column_stack([dH_orb[ax] @ P[:, j] + gspin * muB * apply_spin_operator(P[:, j], S_SPIN[ax])
                              for j in range(2)])
        Mraw = P.conj().T @ DP
        info[f"herm_err_{ax}"] = np.abs(Mraw - Mraw.conj().T).max() / np.abs(Mraw).max()
        M[ax] = 0.5 * (Mraw + Mraw.conj().T)             # eV/T
    return M, info


def g_tensor(M):
    # G[alpha, beta] = Tr(M_alpha sigma_beta) / muB
    return np.array([[np.trace(M[al] @ PAULI[be]).real / muB for be in AX] for al in AX])


results = {}
for lab, E, th in CASES:
    t0 = time.time()
    M, info = zeeman_matrices(E, th)
    results[lab] = (M, g_tensor(M), info)
    print(f"{lab:>12s}: theta = {th:5.1f} deg done ({time.time() - t0:.0f} s): E_pair = {info['E_pair']:.6f} eV, "
          f"Kramers splitting = {info['split0']:.1e} eV, gap to next = {info['gap_next']*1e3:.3f} meV, "
          f"<S_z> = {np.round(info['Sz_pair'], 4)}")

       E = 0: theta =   0.0 deg done (38 s): E_pair = -0.006128 eV, Kramers splitting = 8.9e-16 eV, gap to next = 1.921 meV, <S_z> = [ 0.4996 -0.4996]


      E || y: theta =  90.0 deg done (31 s): E_pair = 0.034326 eV, Kramers splitting = 4.2e-17 eV, gap to next = 1.457 meV, <S_z> = [ 0.3233 -0.3233]


      E || z: theta =   0.0 deg done (28 s): E_pair = 0.053280 eV, Kramers splitting = 6.9e-17 eV, gap to next = 11.974 meV, <S_z> = [ 0.4995 -0.4995]


  theta=83.2: theta =  83.2 deg done (31 s): E_pair = 0.038196 eV, Kramers splitting = 2.1e-16 eV, gap to next = 4.802 meV, <S_z> = [ 0.4673 -0.4673]


## Check 1: is $M_\alpha \propto \sigma_\alpha$?

For every $M_\alpha$ we print the matrix, its decomposition $M_\alpha/\mu_B = \tfrac12(c_0\mathbb 1 + \sum_\beta g_{\alpha\beta}\sigma_\beta)$, the reconstruction error, and the leakage $\max_{\beta\ne\alpha}|g_{\alpha\beta}|/|\mathbf g_\alpha|$.

In [5]:
TOL = 1e-2   # leakage tolerance (fraction of |g_alpha|)

def report(lab, E, th):
    M, G, info = results[lab]
    print("=" * 90)
    print(f"{lab}:  |E| = {E} V/um  (theta = {th} deg from +z toward +y)")
    print("=" * 90)
    ok_all = True
    for i, al in enumerate(AX):
        Mal = M[al] / muB
        c0 = np.trace(Mal).real
        recon = 0.5 * (c0 * np.eye(2) + sum(G[i, j] * PAULI[be] for j, be in enumerate(AX)))
        gnorm = np.linalg.norm(G[i])
        leak = max(abs(G[i, j]) for j in range(3) if j != i) / gnorm
        ok = leak < TOL
        ok_all &= ok
        print(f"\nM_{al}/muB =\n{Mal}")
        print(f"  Tr M_{al}/muB = {c0:+.2e}   recon. error = {np.abs(recon - Mal).max():.1e}   "
              f"(M - M^dag)/|M| before symmetrization = {info['herm_err_' + al]:.1e}")
        print(f"  g_{al}x, g_{al}y, g_{al}z = {G[i, 0]:+.5f}, {G[i, 1]:+.5f}, {G[i, 2]:+.5f}   "
              f"|g_{al}| = {gnorm:.5f}")
        print(f"  leakage max|g_{al}b|/|g_{al}| (b != {al}) = {leak:.2e}  ->  "
              f"M_{al} ∝ sigma_{al}: {'YES' if ok else 'NO'}")
    return ok_all


summary_prop = {lab: report(lab, E, th) for lab, E, th in CASES}

E = 0:  |E| = 0.0 V/um  (theta = 0.0 deg from +z toward +y)

M_x/muB =
[[-0.     +0.j -0.02465+0.j]
 [-0.02465-0.j  0.     +0.j]]
  Tr M_x/muB = -2.27e-16   recon. error = 0.0e+00   (M - M^dag)/|M| before symmetrization = 1.9e-15
  g_xx, g_xy, g_xz = -0.04929, -0.00000, -0.00000   |g_x| = 0.04929
  leakage max|g_xb|/|g_x| (b != x) = 7.38e-14  ->  M_x ∝ sigma_x: YES

M_y/muB =
[[ 0.+0.j       0.-0.21993j]
 [ 0.+0.21993j -0.+0.j     ]]
  Tr M_y/muB = +1.07e-15   recon. error = 2.0e-31   (M - M^dag)/|M| before symmetrization = 1.1e-15
  g_yx, g_yy, g_yz = +0.00000, +0.43985, +0.00000   |g_y| = 0.43985
  leakage max|g_yb|/|g_y| (b != y) = 8.17e-15  ->  M_y ∝ sigma_y: YES

M_z/muB =
[[-7.88414+0.j -0.     -0.j]
 [-0.     +0.j  7.88414+0.j]]
  Tr M_z/muB = -3.64e-14   recon. error = 8.9e-16   (M - M^dag)/|M| before symmetrization = 4.4e-17
  g_zx, g_zy, g_zz = -0.00000, +0.00000, -15.76828   |g_z| = 15.76828
  leakage max|g_zb|/|g_z| (b != z) = 1.12e-15  ->  M_z ∝ sigma_z: YES
E || y:  |E| =

## Check 2 (basis-independent): are $x,y,z$ principal axes?

$\cos\angle(\mathbf g_\alpha, \mathbf g_\beta)$ for $\alpha \ne \beta$ should vanish. If it does, the projected matrices are $\propto\sigma_x,\sigma_y,\sigma_z$ in some pseudospin basis, even if the `psi_new_basis` convention does not pick it. Also compared: $|\mathbf g_\alpha|$ (basis-invariant) vs. $|g_{\alpha\alpha}|$.

In [6]:
for lab, E, th in CASES:
    M, G, info = results[lab]
    gn = np.linalg.norm(G, axis=1)
    cos = (G @ G.T) / np.outer(gn, gn)
    print(f"{lab:>12s}:  cos(g_x,g_y) = {cos[0,1]:+.2e}, cos(g_x,g_z) = {cos[0,2]:+.2e}, "
          f"cos(g_y,g_z) = {cos[1,2]:+.2e}")
    print(f"     |g_x|,|g_y|,|g_z| = {np.round(gn, 5)}   |g_xx|,|g_yy|,|g_zz| = {np.round(np.abs(np.diag(G)), 5)}")

       E = 0:  cos(g_x,g_y) = -8.18e-14, cos(g_x,g_z) = +4.42e-14, cos(g_y,g_z) = -7.71e-15
     |g_x|,|g_y|,|g_z| = [ 0.04929  0.43985 15.76828]   |g_xx|,|g_yy|,|g_zz| = [ 0.04929  0.43985 15.76828]
      E || y:  cos(g_x,g_y) = -7.89e-14, cos(g_x,g_z) = +1.00e-12, cos(g_y,g_z) = -1.50e-15
     |g_x|,|g_y|,|g_z| = [ 0.19401  3.31993 13.62524]   |g_xx|,|g_yy|,|g_zz| = [ 0.19401  3.31993 13.62524]
      E || z:  cos(g_x,g_y) = -2.22e-15, cos(g_x,g_z) = -8.02e-15, cos(g_y,g_z) = -3.39e-14
     |g_x|,|g_y|,|g_z| = [ 0.13438  0.36667 11.04379]   |g_xx|,|g_yy|,|g_zz| = [ 0.13438  0.36667 11.04379]
  theta=83.2:  cos(g_x,g_y) = +8.75e-15, cos(g_x,g_z) = -3.12e-15, cos(g_y,g_z) = -9.88e-01
     |g_x|,|g_y|,|g_z| = [ 0.06589  6.60536 17.50714]   |g_xx|,|g_yy|,|g_zz| = [ 0.06589  0.63411 17.47536]


## Check 3: cross-check with the stored matrices

* `EDSR/M_matrix/M*_...npy`: exported by `EDSR/M_matrix/export_M*.py` after commit 30644f9 (Peierls sign fixed to $e^{-i\ldots}$).
* `EDSR/import_data/M*_...npy`: the copies `main_EDSR_sphere_map.py` reads. **The `Mx` files there predate the Peierls-sign fix**, so they are expected to disagree for $M_x$.

Same N = 100, L = 300 Å and basis convention, so a direct comparison is valid. Only runs for N = 100.

In [7]:
if N == 100:
    for sub in ("M_matrix", "import_data"):
        print(f"--- EDSR/{sub} ---")
        for lab, E, th in CASES[1:]:
            M = results[lab][0]
            if SPIN_UP_FIRST:      # stored files use the raw psi_new_basis order -> undo the swap
                sx = PAULI["x"]
                M = {k: sx @ m @ sx for k, m in M.items()}
            for ax in AX:
                fn = os.path.join("..", "..", "EDSR", sub, f"M{ax}_Eth{th:.2f}_{E_V_um:.3f}Vum_N100_L300.npy")
                if os.path.exists(fn):
                    Mst = np.load(fn)
                    print(f"theta = {th:5.1f}  M_{ax}: max|M_new - M_stored|/max|M_stored| = "
                          f"{np.abs(M[ax] - Mst).max() / np.abs(Mst).max():.2e}")
else:
    print("N != 100: skipped")

--- EDSR/M_matrix ---
theta =  90.0  M_x: max|M_new - M_stored|/max|M_stored| = 1.34e-12
theta =  90.0  M_y: max|M_new - M_stored|/max|M_stored| = 2.48e-14
theta =  90.0  M_z: max|M_new - M_stored|/max|M_stored| = 4.94e-14
theta =   0.0  M_x: max|M_new - M_stored|/max|M_stored| = 4.54e-14
theta =   0.0  M_y: max|M_new - M_stored|/max|M_stored| = 3.22e-14
theta =   0.0  M_z: max|M_new - M_stored|/max|M_stored| = 2.71e-15
theta =  83.2  M_x: max|M_new - M_stored|/max|M_stored| = 4.01e-13
theta =  83.2  M_y: max|M_new - M_stored|/max|M_stored| = 1.13e-14
theta =  83.2  M_z: max|M_new - M_stored|/max|M_stored| = 1.07e-14
--- EDSR/import_data ---
theta =  90.0  M_x: max|M_new - M_stored|/max|M_stored| = 1.16e+00
theta =  90.0  M_y: max|M_new - M_stored|/max|M_stored| = 5.68e-05
theta =  90.0  M_z: max|M_new - M_stored|/max|M_stored| = 1.85e-03
theta =   0.0  M_x: max|M_new - M_stored|/max|M_stored| = 1.81e+00
theta =   0.0  M_y: max|M_new - M_stored|/max|M_stored| = 2.06e-03
theta =   0.0  

## Summary: g-tensor and signs of $g_{ii}$ (|E| = 6 V/µm)

In [8]:
rows = []
print(f"{'direction':>14s} {'g_xx':>9s} {'g_yy':>9s} {'g_zz':>9s} {'sgn(xx,yy,zz)':>14s} "
      f"{'sgn det G':>9s} {'det G':>10s} {'max offdiag/|g|':>16s} {'M∝σ':>5s}")
for lab, E, th in CASES:
    M, G, info = results[lab]
    d = np.diag(G)
    off = max(abs(G[i, j]) / np.linalg.norm(G[i]) for i in range(3) for j in range(3) if i != j)
    sg = "".join("+" if x > 0 else "-" for x in d)
    detG = np.linalg.det(G)
    print(f"{lab:>14s} {d[0]:+9.4f} {d[1]:+9.4f} {d[2]:+9.4f} {sg:>14s} {'+' if detG > 0 else '-':>9s} "
          f"{detG:+10.4f} {off:16.2e} {'yes' if summary_prop[lab] else 'no':>5s}")
    rows.append([E, th, *G.ravel(), detG])

out = f"g_tensor_E{E_V_um:.1f}Vum_N{N}_L{int(L)}.dat"
np.savetxt(out, np.array(rows),
           header="E(V/um)  theta(deg)  gxx gxy gxz  gyx gyy gyz  gzx gzy gzz  detG   "
                  f"[G[a,b] = Tr(M_a sigma_b)/muB; basis: psi_new_basis, SPIN_UP_FIRST={SPIN_UP_FIRST}]")
print(f"\nSaved {out}")
print("\nFull g-tensors G[alpha, beta]:")
for lab, E, th in CASES:
    print(f"{lab}:\n{results[lab][1]}")

     direction      g_xx      g_yy      g_zz  sgn(xx,yy,zz) sgn det G      det G  max offdiag/|g|   M∝σ
         E = 0   -0.0493   +0.4399  -15.7683            -+-         +    +0.3419         7.38e-14   yes
        E || y   -0.1940   +3.3199  -13.6252            -+-         +    +8.7761         9.96e-13   yes
        E || z   +0.1344   -0.3667  -11.0438            +--         +    +0.5442         3.40e-14   yes
    theta=83.2   +0.0659   +0.6341  -17.4754            ++-         -    -1.1869         9.95e-01    no

Saved g_tensor_E6.0Vum_N100_L300.dat

Full g-tensors G[alpha, beta]:
E = 0:
[[ -0.04929  -0.       -0.     ]
 [  0.        0.43985   0.     ]
 [ -0.        0.      -15.76828]]
E || y:
[[ -0.19401  -0.       -0.     ]
 [ -0.        3.31993  -0.     ]
 [ -0.       -0.      -13.62524]]
E || z:
[[  0.13438   0.        0.     ]
 [ -0.       -0.36667   0.     ]
 [  0.       -0.      -11.04379]]
theta=83.2:
[[  0.06589   0.        0.     ]
 [  0.        0.63411   6.57485]
 [ -0.   

In [9]:
# expected E = 0 sign pattern: g_xx < 0, g_yy > 0, g_zz < 0
G0 = results["E = 0"][1]
exp = np.array([-1, +1, -1])
print("E = 0 diag(G) =", np.round(np.diag(G0), 5), " signs match (-,+,-):", np.all(np.sign(np.diag(G0)) == exp))
for lab, E, th in CASES[1:]:
    d = np.diag(results[lab][1])
    flips = [ax for ax, s0, s in zip(AX, np.sign(np.diag(G0)), np.sign(d)) if s0 != s]
    print(f"{lab:>12s}: g_ii relative to E = 0 changed sign for: {flips or 'none'}")

E = 0 diag(G) = [ -0.04929   0.43985 -15.76828]  signs match (-,+,-): True
      E || y: g_ii relative to E = 0 changed sign for: none
      E || z: g_ii relative to E = 0 changed sign for: ['x', 'y']
  theta=83.2: g_ii relative to E = 0 changed sign for: ['x']


### Notes on the signs
* $\operatorname{sign} g_{zz}$ is set by the ordering of the pair (here column 0 = larger $\langle S_z\rangle$ when `SPIN_UP_FIRST`), and $\operatorname{sign} g_{xx}$ by $\langle1|S_x|2\rangle > 0$. Both are conventions. A $\pi$ pseudospin rotation about one axis flips the other two signs.
* $\operatorname{sign}\det G$ does not depend on the basis. It is the sign that can be measured (e.g. through the direction of Larmor precession or the Berry phase). When $G$ is diagonal it equals $\operatorname{sign}(g_{xx}g_{yy}g_{zz})$.

## Check 4: Rashba matrices $R \propto \sigma$ and their signs

First order in $k_x$ inside the Kramers pair (same basis $P$ as for $M_\alpha$ above):
$$H_R = k_x\,R,\qquad R = P^\dagger \frac{\partial H}{\partial k_x}\Big|_{k_x=0} P = \tfrac12 r_0\mathbb 1 + \sum_\beta r_\beta\,\sigma_\beta,\qquad r_\beta = \tfrac12\mathrm{Tr}(R\,\sigma_\beta)\ \ [\text{eV·Å}].$$

$\partial H/\partial k_x$ is built from `dgso` exactly as in `rashba_with_E_pymablock_kwant.py`. As a cross-check, the script's own `compute_H11` (pymablock, $H^{(1)}$ coefficient) is called as well. It uses the raw `psi_new_basis` order (lower $\langle S_z\rangle$ first), so its result is compared after $\sigma_x R\,\sigma_x$ when `SPIN_UP_FIRST`. Same for the stored `EDSR/R_matrix` files.

Expected for a wire along $x$: $H_R \propto (\boldsymbol{\mathcal E}\times\mathbf k)\cdot\boldsymbol\sigma$, i.e. $\mathcal E\parallel y \Rightarrow \sigma_z$, $\mathcal E\parallel z \Rightarrow \sigma_y$, and $R = 0$ at $\mathcal E = 0$ (2H-Ge is centrosymmetric).

**Basis-independent sign.** Under a pseudospin rotation both $r_\beta$ and $g_{\alpha\beta}$ rotate on the index $\beta$. The equivalent spin-orbit field per unit $k_x$,
$$\mathbf B_{SO} = \frac{2}{\mu_B}\,G^{-T}\mathbf r\quad[\text{T·Å}],\qquad \text{defined by }\ \tfrac{\mu_B}{2}\,\mathbf B_{SO}\!\cdot G\,\boldsymbol\sigma = \mathbf r\cdot\boldsymbol\sigma,$$
is therefore convention-free. Its signs are the ones to record.

In [10]:
from Hamiltonian_mathematica_v2 import dgso
import rashba_with_E_pymablock_kwant as rb

# dH/dkx at kx = 0 on the same lattice / site order as H0
def make_dHdkx():
    blocks = {(dy, dz): np.asarray(dgso(a, 0.0, dy, dz), complex)
              for dy in (-1, 0, 1) for dz in (-1, 0, 1)}
    syst = kwant.Builder()
    lat = kwant.lattice.square(norbs=Nband)
    syst[(lat(y, z) for y in range(Ny) for z in range(Nz))] = blocks[(0, 0)]
    for d in [(1, 0), (0, 1), (1, 1), (1, -1)]:
        syst[kwant.builder.HoppingKind(d, lat, lat)] = (lambda s1, s2, B=blocks:
                                                        B[(int(s1.tag[0] - s2.tag[0]), int(s1.tag[1] - s2.tag[1]))])
    fs = syst.finalized()
    assert np.array_equal(tags_of(fs), TAGS)
    return fs.hamiltonian_submatrix(sparse=True).tocsr()

V_kx = make_dHdkx()
print(f"||V_kx - V_kx^dag|| = {abs(V_kx - V_kx.getH()).max():.1e}")
# cross-check: identical to the matrix the Rashba script builds
V_rb = rb.make_system_kx(0.0).hamiltonian_submatrix(sparse=True)
print(f"max|V_kx - V_kx(rashba script)| = {abs(V_kx - V_rb).max():.1e}")


def rashba_vector(R):
    # R = r0/2 * 1 + sum_b r_b sigma_b
    return np.array([np.trace(R @ PAULI[b]).real / 2 for b in AX])


SX2 = PAULI["x"]
rashba = {}
for lab, E, th in CASES:
    M, G, info = results[lab]
    P = info["P"]
    Rraw = P.conj().T @ (V_kx @ P)
    herm = np.abs(Rraw - Rraw.conj().T).max() / max(np.abs(Rraw).max(), 1e-30)
    R = 0.5 * (Rraw + Rraw.conj().T)
    r = rashba_vector(R)
    r0 = np.trace(R).real
    # Rashba script (pymablock, raw psi_new_basis order)
    t0 = time.time()
    H11 = rb.compute_H11(E * 1e-4, np.deg2rad(th), V_kx)
    if np.ndim(H11) == 0:                     # pymablock returns its `Zero` object for a vanishing term
        H11 = np.zeros((2, 2), complex)
    if SPIN_UP_FIRST:
        H11 = SX2 @ H11 @ SX2
    d_pmb = np.abs(H11 - R).max() / max(np.abs(R).max(), 1e-12)
    rashba[lab] = dict(R=R, r=r, r0=r0, herm=herm, H11=H11, d_pmb=d_pmb)
    print("=" * 90)
    print(f"{lab}:  |E| = {E} V/um, theta = {th} deg   (pymablock {time.time() - t0:.0f} s)")
    print(f"R (eV A) =\n{R}")
    print(f"  r0 = {r0:+.2e}   r_x, r_y, r_z = {r[0]:+.5f}, {r[1]:+.5f}, {r[2]:+.5f}   |r| = {np.linalg.norm(r):.5f} eV A")
    print(f"  (R - R^dag)/|R| before symmetrization = {herm:.1e}")
    print(f"  pymablock H11 (rashba script):  max|H11 - R|/|R| = {d_pmb:.1e}   (H11 =\n{np.round(H11, 5)})")

||V_kx - V_kx^dag|| = 0.0e+00


max|V_kx - V_kx(rashba script)| = 0.0e+00


E = 0:  |E| = 0.0 V/um, theta = 0.0 deg   (pymablock 46 s)
R (eV A) =
[[0.+0.j 0.+0.j]
 [0.-0.j 0.+0.j]]
  r0 = +1.36e-15   r_x, r_y, r_z = +0.00000, -0.00000, +0.00000   |r| = 0.00000 eV A
  (R - R^dag)/|R| before symmetrization = 1.6e-03
  pymablock H11 (rashba script):  max|H11 - R|/|R| = 1.2e-03   (H11 =
[[0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]])


E || y:  |E| = 6.0 V/um, theta = 90.0 deg   (pymablock 39 s)
R (eV A) =
[[-0.11216+0.j -0.     +0.j]
 [-0.     -0.j  0.11216+0.j]]
  r0 = -1.96e-14   r_x, r_y, r_z = -0.00000, -0.00000, -0.11216   |r| = 0.11216 eV A
  (R - R^dag)/|R| before symmetrization = 1.6e-16
  pymablock H11 (rashba script):  max|H11 - R|/|R| = 2.6e-13   (H11 =
[[-0.11216+0.j -0.     -0.j]
 [-0.     +0.j  0.11216-0.j]])


E || z:  |E| = 6.0 V/um, theta = 0.0 deg   (pymablock 28 s)
R (eV A) =
[[-0.+0.j       0.-0.00958j]
 [ 0.+0.00958j  0.+0.j     ]]
  r0 = +2.79e-17   r_x, r_y, r_z = +0.00000, +0.00958, -0.00000   |r| = 0.00958 eV A
  (R - R^dag)/|R| before symmetrization = 1.3e-15
  pymablock H11 (rashba script):  max|H11 - R|/|R| = 3.4e-14   (H11 =
[[-0.-0.j       0.-0.00958j]
 [ 0.+0.00958j -0.-0.j     ]])


theta=83.2:  |E| = 6.0 V/um, theta = 83.2 deg   (pymablock 34 s)
R (eV A) =
[[-0.14771+0.j      -0.     -0.01374j]
 [-0.     +0.01374j  0.14771+0.j     ]]
  r0 = -6.38e-16   r_x, r_y, r_z = -0.00000, +0.01374, -0.14771   |r| = 0.14835 eV A
  (R - R^dag)/|R| before symmetrization = 1.2e-16
  pymablock H11 (rashba script):  max|H11 - R|/|R| = 1.1e-14   (H11 =
[[-0.14771+0.j       0.     -0.01374j]
 [ 0.     +0.01374j  0.14771+0.j     ]])


In [11]:
# Is R ∝ a single sigma? leakage + stored EDSR files
TOLR = 1e-2
print(f"{'case':>12s} {'r_x':>9s} {'r_y':>9s} {'r_z':>9s}  {'dominant':>8s} {'leakage':>9s} {'R∝σ':>5s}   stored EDSR/R_matrix (rel. diff)")
for lab, E, th in CASES:
    r = rashba[lab]["r"]
    rn = np.linalg.norm(r)
    if rn < 1e-8:
        dom, leak, ok = "-", 0.0, "R=0"
    else:
        i = int(np.argmax(np.abs(r)))
        dom = f"sigma_{AX[i]}"
        leak = max(abs(r[j]) for j in range(3) if j != i) / rn
        ok = "yes" if leak < TOLR else "no"
    stored = ""
    fn = os.path.join("..", "..", "EDSR", "R_matrix", f"R_mat_th{th:.2f}_E{E_V_um:.2f}_N100_L300.npy")
    if E > 0 and N == 100 and os.path.exists(fn):
        Rst = np.load(fn)
        Rcmp = SX2 @ rashba[lab]["R"] @ SX2 if SPIN_UP_FIRST else rashba[lab]["R"]
        stored = f"{np.abs(Rcmp - Rst).max() / np.abs(Rst).max():.1e}"
    print(f"{lab:>12s} {r[0]:+9.5f} {r[1]:+9.5f} {r[2]:+9.5f}  {dom:>8s} {leak:9.2e} {ok:>5s}   {stored}")

        case       r_x       r_y       r_z  dominant   leakage   R∝σ   stored EDSR/R_matrix (rel. diff)
       E = 0  +0.00000  -0.00000  +0.00000         -  0.00e+00   R=0   
      E || y  -0.00000  -0.00000  -0.11216   sigma_z  4.24e-15   yes   2.1e-13
      E || z  +0.00000  +0.00958  -0.00000   sigma_y  2.58e-14   yes   4.6e-14
  theta=83.2  -0.00000  +0.01374  -0.14771   sigma_z  9.26e-02    no   8.3e-15


In [12]:
# Signs: r_b (convention dependent) and B_SO = (2/muB) G^{-T} r (convention free)
rows_R = []
print(f"{'case':>12s} {'sgn(r_x,r_y,r_z)':>17s} {'sgn(g_bb r_b)':>14s}   B_SO = (2/muB) G^-T r  [T A]        sgn(B_SO)")
for lab, E, th in CASES:
    G = results[lab][1]
    r = rashba[lab]["r"]
    rn = np.linalg.norm(r)
    sgn = lambda v: "".join("0" if abs(x) < 1e-6 * max(rn, 1e-30) or rn < 1e-8 else ("+" if x > 0 else "-") for x in v)
    BSO = (2 / muB) * np.linalg.solve(G.T, r)
    gr = np.diag(G) * r
    print(f"{lab:>12s} {sgn(r):>17s} {sgn(gr):>14s}   ({BSO[0]:+10.3e}, {BSO[1]:+10.3e}, {BSO[2]:+10.3e})   {sgn(BSO)}")
    rows_R.append([E, th, *r, rashba[lab]["r0"], *BSO])

out_R = f"rashba_E{E_V_um:.1f}Vum_N{N}_L{int(L)}.dat"
np.savetxt(out_R, np.array(rows_R),
           header="E(V/um)  theta(deg)  r_x r_y r_z (eV A)  r0 (eV A)  BSO_x BSO_y BSO_z (T A)   "
                  f"[R = r0/2 + r.sigma, r_b = Tr(R sigma_b)/2, R = P^dag dH/dkx P; BSO = (2/muB) G^-T r; "
                  f"basis: psi_new_basis, SPIN_UP_FIRST={SPIN_UP_FIRST}]")
print(f"\nSaved {out_R}")

        case  sgn(r_x,r_y,r_z)  sgn(g_bb r_b)   B_SO = (2/muB) G^-T r  [T A]        sgn(B_SO)
       E = 0               000            000   (-6.835e-11, -1.004e-11, -1.055e-12)   000
      E || y               00-            00+   (-1.109e-11, -1.059e-12, +2.844e+02)   00+
      E || z               0+0            0-0   (+7.335e-14, -9.030e+02, -2.459e-13)   0-0
  theta=83.2               0+-            0++   (+2.086e-13, +1.618e+02, +3.530e+02)   0++

Saved rashba_E6.0Vum_N100_L300.dat
